# P07 — Traducción automática neuronal aprendiendo conjuntamente a alinear y traducir

## 1. Título y paper

**Paper:** *Neural Machine Translation by Jointly Learning to Align and Translate*  
**Autoría:** Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio  
**Año y venue:** 2014 · arXiv:1409.0473 · ICLR 2015  
**Nivel:** L3 · **Motor:** `bahdanau`  
**Ficha completa:** [`P07_attention_bahdanau`](../../papers/foundational/P07_attention_bahdanau/README.md)

**Hito:** Nace la atención: el decodificador deja de depender de un único vector y consulta toda la entrada en cada paso.

- [arXiv:1409.0473](https://arxiv.org/abs/1409.0473)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Comprimir una frase entera en un vector fijo degrada la traducción de frases largas: es un cuello de botella de información.
2. Ejecutar una implementación mínima de la propuesta: Un vector de contexto distinto por paso de salida, calculado como suma ponderada de los estados del codificador con pesos aprendidos (atención aditiva).
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P06


## 4. Intuición

En vez de memorizar la frase entera y cerrar los ojos, el traductor deja el texto original sobre la mesa y, para cada palabra que escribe, vuelve a mirar la parte que le hace falta.


## 5. Concepto mínimo

```text
e_ij = vᵀ·tanh(W·s_{i−1} + U·h_j)      puntuación de compatibilidad (aditiva)
α_ij = softmax_j(e_ij)                  pesos que suman 1
c_i  = Σ_j α_ij · h_j                   vector de contexto DISTINTO en cada paso i
```

El cuello de botella desaparece: `c_i` se recalcula en cada paso a partir de todos los estados del codificador.


## 6. Código explicado

El motor aprende los 18 parámetros de la atención aditiva por descenso de gradiente y muestra la matriz α.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('bahdanau', seed=7)['result']
print('parámetros:', r['parametros'], '· aciertos:', r['aciertos_de_alineacion'])
for fila in r['alignment']:
    print(f"{fila['target']:<8} → {fila['argmax']:<8} α={fila['alpha']} H={fila['entropia']}")

## 7. Predicción antes de ejecutar

1. ¿Sumarán exactamente 1 los pesos α de cada fila? ¿Por qué?
2. ¿Será la entropía alta (atención repartida) o baja (atención concentrada) tras entrenar?
3. Si la atención acierta la alineación, ¿demuestra eso que el modelo «entiende» la frase?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('bahdanau', seed=semilla)['result']
    entropias = [f['entropia'] for f in r['alignment']]
    print(f"semilla {semilla:>2} · aciertos {r['aciertos_de_alineacion']} "
          f"· entropía media {sum(entropias)/len(entropias):.3f} "
          f"· pérdida final {r['perdida'][-1]['loss']}")

## 9. Salida interpretable

La entropía baja (cerca de 0) significa que α se concentra casi todo en una posición: la alineación es nítida. `suma_alpha = 1.0` en cada fila confirma que softmax produce una distribución de probabilidad sobre las posiciones de entrada.


## 10. Comentario pedagógico

Aquí la alineación está supervisada para que el mecanismo se vea. En el paper **nadie etiqueta la alineación**: emerge al entrenar solo la traducción. Esa es la parte notable, y conviene no atribuir a este notebook un mérito que corresponde al paper.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer la matriz de atención como una explicación causal («el modelo se fijó en X porque X importa»).


In [ ]:
print('Afirmación tentadora: «α muestra en qué se fijó el modelo, luego explica su decisión».')
print('Jain y Wallace (2019) mostraron que se pueden construir distribuciones de atención')
print('muy distintas que producen la MISMA salida. Correlación ≠ explicación.')

## 12. Corrección

Enunciado defendible sobre lo que la atención sí aporta:


In [ ]:
defendible = [
    'α es un peso de mezcla, verificable y que suma 1',
    'α elimina el cuello de botella del vector fijo (eso sí es causal en la arquitectura)',
    'α es una PISTA de interpretación, no una explicación del proceso interno',
]
for linea in defendible:
    print('-', linea)

## 13. Desafío guiado

Comprueba qué ocurre si eliminas el softmax y usas los scores crudos como pesos.


In [ ]:
scores = [2.0, 1.0, 0.5, -3.0]
suma_cruda = sum(scores)
print('pesos crudos normalizados:', [round(s / suma_cruda, 3) for s in scores])
print('→ hay pesos NEGATIVOS y la suma se rompe si los scores suman ~0')
import math
exp = [math.exp(s) for s in scores]
print('softmax                  :', [round(e / sum(exp), 3) for e in exp])

## 14. Desafío autónomo

Entrena atención aditiva sin supervisión de alineación: solo con la pérdida de predecir el token siguiente en un corpus paralelo de juguete. Comprueba si la alineación emerge sola y reporta cuántos ejemplos hicieron falta.


## 15. Evidencia de aprendizaje

Guarda la matriz α, la entropía por fila y tu enunciado sobre qué se puede y qué no se puede concluir de esa matriz.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P07_attention_bahdanau/README.md) · evaluación formal: [`assessments/papers/P07_attention_bahdanau.md`](../../assessments/papers/P07_attention_bahdanau.md)


## 16. Cierre

Si la atención resuelve el acceso a toda la entrada… ¿para qué sigue haciendo falta la recurrencia? Esa pregunta es el título del paper siguiente.


## 17. Conexión con el siguiente hito

- P08

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
